# AI reviewing security configurations  (MODEL SOLUTION, instructor copy, do not distribute)

# Coding problem, 1.5.a: AI reviewing security configurations

**Skill:** Mitigate Risk, Collaborate · **Maps to:** 1.5.A · **Language:** Python 3 (pytest)

## Problem
Build the rule-engine core of an AI-assisted configuration auditor. Given a list of configuration
file lines (as a real tool would read from `/etc/ssh/sshd_config` or a service config), return
the sorted set of security finding tags for every insecure directive detected. This mirrors exactly
what tools like Lynis and OpenSCAP do at their rule-evaluation layer.

## Function contract
Implement `solve(config_lines: list[str]) -> list[str]` in the code editor
(see the Reference section below). Return a sorted, de-duplicated list of finding tag strings.

## Input
A `list[str]`, each element one line from a config file. Lines may be blank, start with `#`
(comments), or contain leading/trailing whitespace. Matching is case-insensitive.

Example input:
```python
[
    "# sshd_config for web01",
    "Port 22",
    "PermitRootLogin yes",
    "PasswordAuthentication yes",
    "X11Forwarding yes",
    "",
]
```

## Output
A sorted `list[str]` of finding tags. Tags are drawn from:
- `"insecure:PermitRootLogin"`
- `"insecure:PasswordAuthentication"`
- `"insecure:Port23"`
- `"insecure:SSLDisabled"`
- `"insecure:X11Forwarding"`
- `"insecure:PermitEmptyPasswords"`

## Examples
```
["PermitRootLogin yes", "Port 22"]
    -> ["insecure:PermitRootLogin"]

["PasswordAuthentication yes", "X11Forwarding yes", "PermitRootLogin yes"]
    -> ["insecure:PasswordAuthentication", "insecure:PermitRootLogin", "insecure:X11Forwarding"]

["port 23", "ssl false"]
    -> ["insecure:Port23", "insecure:SSLDisabled"]

["# all defaults", "Port 22", "PermitRootLogin no", ""]
    -> []
```

## Constraints
- Deterministic, offline, standard library only.
- Must pass all cases in the tests (visible + hidden).

## Reference
Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

### Entry point
```python
def solve(config_lines: list[str]) -> list[str]
```

### Input (`config_lines`)
A list of strings, each representing one line of a configuration file (e.g., SSH `sshd_config`
or a generic service config). Lines may include whitespace, inline comments, or blank lines.
Matching is case-insensitive and strips leading/trailing whitespace per line.

### Output (`result`)
A **sorted, de-duplicated** `list[str]` of finding tags drawn from the set below.
Return an empty list if no insecure settings are found.

### Rule set (case-insensitive substring match on each stripped line)

| Tag returned                      | Trigger substring(s)                              | Why it is insecure                          |
|-----------------------------------|---------------------------------------------------|---------------------------------------------|
| `"insecure:PermitRootLogin"`      | `permitrootlogin yes`                             | Allows direct root login over SSH           |
| `"insecure:PasswordAuthentication"` | `passwordauthentication yes`                    | Enables brute-force password attacks        |
| `"insecure:Port23"`               | `port 23`                                         | Port 23 = Telnet; cleartext credentials     |
| `"insecure:SSLDisabled"`          | `ssl false` OR `ssl off` OR `tls false` OR `tls off` | Disables transport encryption           |
| `"insecure:X11Forwarding"`        | `x11forwarding yes`                               | Unnecessary attack surface                  |
| `"insecure:PermitEmptyPasswords"` | `permitemptypasswords yes`                        | Allows accounts with no password            |

### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Matching is case-insensitive. Output always sorted ascending (lexicographic) and de-duplicated.
- A single line may trigger at most one rule (rules are mutually exclusive by tag).
- Blank lines and lines whose stripped form starts with `#` are ignored.
- A config_lines value of `[]` returns `[]`.

### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).


In [ ]:
# Reference solution
"""Reference solution for 1.5.a. MUST pass 100% of the tests.
Also the default 'blackbox' implementation students can swap out (see the hidden grading tests)."""
from typing import List

# Rule set: each entry is (tag, list_of_trigger_substrings).
# A line triggers the tag if ANY trigger substring matches (case-insensitive).
RULES = [
    ("insecure:PermitRootLogin",      ["permitrootlogin yes"]),
    ("insecure:PasswordAuthentication", ["passwordauthentication yes"]),
    ("insecure:Port23",               ["port 23"]),
    ("insecure:SSLDisabled",          ["ssl false", "ssl off", "tls false", "tls off"]),
    ("insecure:X11Forwarding",        ["x11forwarding yes"]),
    ("insecure:PermitEmptyPasswords", ["permitemptypasswords yes"]),
]


def solve(config_lines: List[str]) -> List[str]:
    """Return sorted, de-duplicated security finding tags for insecure config directives."""
    findings = set()
    for raw in config_lines:
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        low = line.lower()
        for tag, triggers in RULES:
            if any(t in low for t in triggers):
                findings.add(tag)
    return sorted(findings)


## Self-check tests
Run the cell below to check your `solve` implementation above.

In [ ]:
import __main__ as impl  # the functions defined above

# ---- visible cases -------------------------------------------------------
def test_single_root_login():
    lines = ["PermitRootLogin yes", "Port 22"]
    assert impl.solve(lines) == ["insecure:PermitRootLogin"]


def test_multiple_findings():
    lines = ["PasswordAuthentication yes", "X11Forwarding yes", "PermitRootLogin yes"]
    assert impl.solve(lines) == [
        "insecure:PasswordAuthentication",
        "insecure:PermitRootLogin",
        "insecure:X11Forwarding",
    ]


def test_port23_and_ssl():
    lines = ["port 23", "ssl false"]
    assert impl.solve(lines) == ["insecure:Port23", "insecure:SSLDisabled"]


def test_all_safe():
    lines = ["# all defaults", "Port 22", "PermitRootLogin no", ""]
    assert impl.solve(lines) == []


# ---- hidden / edge cases -------------------------------------------------
def test_empty_input():
    assert impl.solve([]) == []


def test_case_insensitive():
    assert impl.solve(["PERMITROOTLOGIN YES"]) == ["insecure:PermitRootLogin"]


def test_comment_lines_ignored():
    assert impl.solve(["# PermitRootLogin yes"]) == []


def test_blank_lines_ignored():
    assert impl.solve(["", "   ", "\t"]) == []


def test_tls_off_variant():
    assert impl.solve(["tls off"]) == ["insecure:SSLDisabled"]


def test_ssl_off_variant():
    assert impl.solve(["SSL OFF"]) == ["insecure:SSLDisabled"]


def test_permit_empty_passwords():
    assert impl.solve(["PermitEmptyPasswords yes"]) == ["insecure:PermitEmptyPasswords"]


def test_dedup_sorted():
    # two lines both triggering PermitRootLogin should yield one tag
    lines = ["PermitRootLogin yes", "permitrootlogin yes"]
    result = impl.solve(lines)
    assert result == sorted(set(result))
    assert result == ["insecure:PermitRootLogin"]


def test_full_config():
    config = [
        "# /etc/ssh/sshd_config for dev-server",
        "Port 22",
        "PermitRootLogin yes",
        "PasswordAuthentication yes",
        "X11Forwarding yes",
        "PermitEmptyPasswords yes",
        "AllowTcpForwarding yes",
        "",
    ]
    assert impl.solve(config) == [
        "insecure:PasswordAuthentication",
        "insecure:PermitEmptyPasswords",
        "insecure:PermitRootLogin",
        "insecure:X11Forwarding",
    ]


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")
